# MET experiment v4 (GPU, Kaggle)
**Before running:** Settings → Accelerator = **GPU T4 x2**, Internet = **On**; Add Input → your **met-experiment** dataset (**latest version**); HF_TOKEN secret ticked for gated models.

Choose the model **and which test sets to run** in Step 2. Run cells top to bottom through **Step 4**; if it prints `ALL CHECKS PASSED`, click **Save Version → Save & Run All**, then stop the interactive session.

## Step 1: environment

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count())

In [ ]:
!pip install -q transformers==4.46.3 accelerate==1.1.1 spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import glob, os, shutil
# pick the dataset copy that contains the NaN guard (newest code)
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/met/methods.py', recursive=True)
         if 'FloatingPointError' in open(p).read() and os.path.exists(os.path.join(os.path.dirname(os.path.dirname(p)),'data','met_edits.jsonl')) and 'robustrag_decoding' in open(p).read() and 'grounded' in open(p).read()]
assert cands, 'Updated code not found: upload met-experiment.zip as a NEW VERSION of your dataset and re-attach it.'
shutil.copytree(cands[0], '/kaggle/working/exp', dirs_exist_ok=True)
os.chdir('/kaggle/working/exp'); os.makedirs('results', exist_ok=True)
print('code from', cands[0]); print(os.listdir('.'))

## Step 2: choose the model
Qwen2.5 and Gemma-2 **must** use `bfloat16` (float16 overflows). Llama-3.1 and Mistral are fine in either.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'          # or 'mistralai/Mistral-7B-Instruct-v0.3', 'Qwen/Qwen2.5-7B-Instruct'
DTYPE = 'bfloat16'                                # Qwen2.5 / Gemma-2 need 'bfloat16'
RUNS = ['rrag_known', 'cf']                       # any of: 'known','met','retrieval' (all main methods) or
                                                  #   'rrag_known','rrag_met','rrag_retrieval' (RobustRAG baselines only), 'cf' (all 300 counterfactual pairs, all methods),
                                                  #   'edits' (multi-doc legitimate edits), 'retrieval_small'
TAG = MODEL_ID.split('/')[-1]
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN'); print('HF token loaded')
except Exception:
    print('no HF token (fine for Qwen / Mistral)')
print(TAG, DTYPE, RUNS)

## Step 3: download the model (auto-retries on stalls)

In [ ]:
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
from huggingface_hub import snapshot_download
MODEL = None
for attempt in range(20):
    try:
        MODEL = snapshot_download(MODEL_ID, allow_patterns=['*.json','*.safetensors','*.txt','*.model'], max_workers=1,
                                  ignore_patterns=['consolidated*', 'original/*'])  # skip duplicate Mistral/Meta-format weights
        break
    except Exception as e:
        print(f'attempt {attempt+1} failed ({type(e).__name__}), retrying...')
assert MODEL, 'download failed 20 times'
print('MODEL =', MODEL)

## Step 4: safety check (1 fact x 4 poisoning levels)

In [ ]:
for f in glob.glob(f'results/{TAG}_smoke.jsonl'): os.remove(f)
r = subprocess.run(['python','run.py','--model',MODEL,'--data','data/met_known.jsonl','--limit_q','2',
                    '--out',f'results/{TAG}_smoke.jsonl','--dtype',DTYPE,'--device','auto'],capture_output=True,text=True)
print(r.stdout[-2500:]); print(r.stderr[-1500:] if r.returncode else '')
assert r.returncode == 0, 'run.py crashed - see error above'
import json, math
rows = [json.loads(l) for l in open(f'results/{TAG}_smoke.jsonl')]
bad_conf = [x['id'] for x in rows if not math.isfinite(x['prior_conf'])]
answers = [a for x in rows for a in list(x['pred'].values()) + (x['isolated'] or [])]
bang = sum(a.count('!') for a in answers)
print('sample answers:', answers[:8])
print('prior_conf values:', [x['prior_conf'] for x in rows])
assert not bad_conf, f'NaN confidence in {bad_conf}'
assert bang <= 2, f'{bang} stray "!" characters in answers -> precision problem'
print('ALL CHECKS PASSED')

## Step 5: full runs
Float16 7-8B model: known ≈ 45 min, met ≈ 1 h, retrieval ≈ 2 h; RobustRAG-only runs: rrag_known ≈ 40 min, rrag_retrieval ≈ 1.5 h (about 4x longer in bfloat16).

In [ ]:
SPECS = {'known':     ['--data','data/met_known.jsonl'],
         'met':       ['--data','data/met.jsonl','--limit_q','30'],
         'retrieval': ['--data','data/met_retrieval.jsonl','--parts','retrieval'],
         # all 300 Faithfulness-QA counterfactual pairs, main methods + RobustRAG (legitimate-edit test)
         'cf':        ['--data','data/met.jsonl','--parts','counterfactual',
                       '--methods','closed_book,rag,cad,adacad,arr,sdag,vote,pvote,rrag_kw,rrag_dec'],
         # legitimate edits inside a 5-document context (200 questions x orig/edit/update), all methods
         'edits':     ['--data','data/met_edits.jsonl',
                       '--methods','closed_book,rag,cad,adacad,arr,sdag,vote,pvote,rrag_kw,rrag_dec'],
         # smaller real-retrieval run (35 questions per corpus = 105 x 4 injection levels), all methods (for slow bf16 models)
         'retrieval_small': ['--data','data/met_retrieval.jsonl','--parts','retrieval','--limit_q','35',
                       '--methods','closed_book,rag,cad,adacad,arr,sdag,vote,pvote,rrag_kw,rrag_dec']}
RR = ['--methods','closed_book,rrag_kw,rrag_dec']
SPECS.update({'rrag_known': SPECS['known']+RR, 'rrag_met': SPECS['met']+RR, 'rrag_retrieval': SPECS['retrieval']+RR})
for name in RUNS:
    print(f'===== {name} =====', flush=True)
    cmd = ['python','run.py','--model',MODEL,'--out',f'results/{TAG}_{name}.jsonl','--dtype',DTYPE,'--device','auto'] + SPECS[name]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if 'past_key_values' not in line: print(line, end='', flush=True)
    assert p.wait() == 0, f'{name} run failed'

In [ ]:
for name in RUNS:
    subprocess.run(['python','analyze.py',f'results/{TAG}_{name}.jsonl'])
for f in glob.glob(f'results/{TAG}_*.jsonl'): shutil.copy(f, '/kaggle/working/')
print(sorted(os.listdir('/kaggle/working')))